# Load PRD MURA model from MLflow and predict

Clean self-contained Colab notebook. It needs only:
- this notebook;
- reachable MLflow tracking URI;
- one image path, or raw `MURA-v1.1.zip` to extract a sample image.


In [ ]:
VPS_HOST = "<>"
MLFLOW_TRACKING_URI = f"http://{VPS_HOST}:5050"
REGISTERED_MODEL_NAME = "mura_dinov2_transformer"

# Optional: used only to extract one test image for the smoke predict.
MURA_ZIP_PATH = "/content/drive/MyDrive/Project2025/MURA-v1.1.zip"
SAMPLE_IMAGE_PATH = ""  # leave empty to auto-pick from MURA-v1.1.zip
SAMPLE_ANATOMY = ""     # leave empty to infer from path


In [ ]:
%pip -q install "mlflow>=2.15.1,<3" "torch" "torchvision" "transformers>=4.45" "pandas" "numpy" "pillow"


In [ ]:
from pathlib import Path
import os
import zipfile
import shutil

import mlflow
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print("MLflow tracking URI:", mlflow.get_tracking_uri())


In [ ]:
def find_mura_root(extract_dir: Path) -> Path:
    candidates = [extract_dir / "MURA-v1.1", extract_dir]
    candidates.extend([p for p in extract_dir.rglob("*") if p.is_dir() and p.name == "MURA-v1.1"])
    for candidate in candidates:
        if (candidate / "train").is_dir() and (candidate / "valid").is_dir():
            return candidate
    for candidate in extract_dir.rglob("*"):
        if candidate.is_dir() and (candidate / "train").is_dir() and (candidate / "valid").is_dir():
            return candidate
    raise FileNotFoundError("Could not find MURA-v1.1 root")


IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp"}


def is_real_image_file(path: Path) -> bool:
    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        return False
    if path.name.startswith("._") or path.name == ".DS_Store":
        return False
    if "__MACOSX" in path.parts:
        return False
    return True


def infer_anatomy(path: str) -> str:
    for part in Path(path).parts:
        if part.startswith("XR_"):
            return part
    raise ValueError(f"Cannot infer anatomy from {path}")


def get_sample_image() -> tuple[str, str]:
    if SAMPLE_IMAGE_PATH:
        anatomy = SAMPLE_ANATOMY or infer_anatomy(SAMPLE_IMAGE_PATH)
        return SAMPLE_IMAGE_PATH, anatomy

    zip_path = Path(MURA_ZIP_PATH)
    if not zip_path.exists():
        alt = Path("/content/MURA-v1.1.zip")
        if alt.exists():
            zip_path = alt
        else:
            raise FileNotFoundError("Set SAMPLE_IMAGE_PATH or put MURA-v1.1.zip at MURA_ZIP_PATH or /content/MURA-v1.1.zip")

    extract_dir = Path("/content/mura_predict_sample")
    marker = extract_dir / ".done"
    if not marker.exists():
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)
        marker.write_text(str(zip_path))
    root = find_mura_root(extract_dir)
    image_path = next(p for p in (root / "valid").rglob("*") if p.is_file() and is_real_image_file(p))
    return str(image_path), infer_anatomy(str(image_path))

image_path, anatomy = get_sample_image()
print("sample:", image_path, anatomy)


In [ ]:
model = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL_NAME}@prd")
input_df = pd.DataFrame({"image_path": [image_path], "anatomy": [anatomy]})
pred = model.predict(input_df)
display(pd.concat([input_df, pred], axis=1))
